# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd

file1 = pd.read_csv("file1.csv")
file2 = pd.read_csv("file2.csv")
file3 = pd.read_csv("file3.csv")

print("File 1 shape:", file1.shape)
print("File 2 shape:", file2.shape)
print("File 3 shape:", file3.shape)

File 1 shape: (4008, 11)
File 2 shape: (996, 11)
File 3 shape: (7070, 11)


In [2]:
print("File 1 columns:")
print(file1.columns.tolist())

print("\nFile 2 columns:")
print(file2.columns.tolist())

print("\nFile 3 columns:")
print(file3.columns.tolist())

File 1 columns:
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']

File 2 columns:
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']

File 3 columns:
['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']


In [3]:
def clean_column_names(df):
    df = df.copy()

    # Convert names to lowercase and replace spaces with underscores
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # ST and State represent the same information
    df = df.rename(columns={"st": "state"})

    return df

In [8]:
file1 = clean_column_names(file1)
file2 = clean_column_names(file2)
file3 = clean_column_names(file3)

In [9]:
customer_data = pd.concat(
    [file1, file2, file3],
    ignore_index=True
)

In [10]:
print("Combined shape:", customer_data.shape)

customer_data.head()

Combined shape: (12074, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [11]:
print("Combined shape:", customer_data.shape)

print(
    "Completely empty rows:",
    customer_data.isna().all(axis=1).sum()
)

print(
    "Duplicate rows:",
    customer_data.duplicated().sum()
)

print("\nMissing values by column:")
print(customer_data.isna().sum())

Combined shape: (12074, 11)
Completely empty rows: 2937
Duplicate rows: 2939

Missing values by column:
customer                     2937
state                        2937
gender                       3059
education                    2937
customer_lifetime_value      2944
income                       2937
monthly_premium_auto         2937
number_of_open_complaints    2937
policy_type                  2937
vehicle_class                2937
total_claim_amount           2937
dtype: int64


In [12]:
customer_data = customer_data.dropna(how="all")

customer_data = customer_data.drop_duplicates()

customer_data = customer_data.reset_index(drop=True)

In [13]:
print("Cleaned shape:", customer_data.shape)

print(
    "Completely empty rows:",
    customer_data.isna().all(axis=1).sum()
)

print(
    "Duplicate rows:",
    customer_data.duplicated().sum()
)

Cleaned shape: (9134, 11)
Completely empty rows: 0
Duplicate rows: 0


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [14]:
# Standardize state names
customer_data["state"] = customer_data["state"].replace({
    "Cali": "California",
    "AZ": "Arizona",
    "WA": "Washington"
})

# Standardize gender values
customer_data["gender"] = customer_data["gender"].replace({
    "F": "Female",
    "female": "Female",
    "Femal": "Female",
    "M": "Male",
    "Male": "Male",
    "NA": pd.NA
})

# Standardize education values
customer_data["education"] = customer_data["education"].replace({
    "Bachelors": "Bachelor"
})

In [15]:
print("States:")
print(customer_data["state"].value_counts(dropna=False))

print("\nGenders:")
print(customer_data["gender"].value_counts(dropna=False))

print("\nEducation:")
print(customer_data["education"].value_counts(dropna=False))

States:
state
California    3150
Oregon        2601
Arizona       1703
Nevada         882
Washington     798
Name: count, dtype: int64

Genders:
gender
Female    4604
Male      4408
NaN        122
Name: count, dtype: int64

Education:
education
Bachelor                2742
College                 2681
High School or Below    2616
Master                   751
Doctor                   344
Name: count, dtype: int64


In [16]:
# Identify the values originally written as percentages
percentage_rows = (
    customer_data["customer_lifetime_value"]
    .astype("string")
    .str.contains("%", na=False)
)

# Remove the percentage symbol and convert to numbers
customer_data["customer_lifetime_value"] = pd.to_numeric(
    customer_data["customer_lifetime_value"]
    .astype("string")
    .str.replace("%", "", regex=False),
    errors="coerce"
)

# Convert percentage-formatted values to their correct scale
customer_data.loc[
    percentage_rows,
    "customer_lifetime_value"
] /= 100

In [18]:
def clean_complaints(value):

    if pd.isna(value):
        return pd.NA

    value = str(value)

    # For values such as 1/2/00, keep the middle number
    if "/" in value:
        value = value.split("/")[1]

    return pd.to_numeric(value, errors="coerce")

In [19]:
customer_data["number_of_open_complaints"] = (
    customer_data["number_of_open_complaints"]
    .apply(clean_complaints)
    .astype("Int64")
)

In [20]:
print(
    customer_data[
        [
            "customer_lifetime_value",
            "number_of_open_complaints"
        ]
    ].head()
)

print("\nData types:")
print(
    customer_data[
        [
            "customer_lifetime_value",
            "number_of_open_complaints"
        ]
    ].dtypes
)

   customer_lifetime_value  number_of_open_complaints
0                     <NA>                          0
1                6979.5359                          0
2               12887.4317                          0
3                7645.8618                          0
4                5363.0765                          0

Data types:
customer_lifetime_value      Float64
number_of_open_complaints      Int64
dtype: object


In [21]:
print("Missing values by column:")

print(
    customer_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Missing values by column:
gender                       122
customer_lifetime_value        7
customer                       0
state                          0
education                      0
income                         0
monthly_premium_auto           0
number_of_open_complaints      0
policy_type                    0
vehicle_class                  0
total_claim_amount             0
dtype: int64


In [22]:
print("\nColumn data types:")

print(customer_data.dtypes)


Column data types:
customer                      object
state                         object
gender                        object
education                     object
customer_lifetime_value      Float64
income                       float64
monthly_premium_auto         float64
number_of_open_complaints      Int64
policy_type                   object
vehicle_class                 object
total_claim_amount           float64
dtype: object


In [23]:
print("\nCurrent shape:", customer_data.shape)

customer_data.head()


Current shape: (9134, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,<NA>,0.0,1000.0,0,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,Female,Bachelor,6979.5359,0.0,94.0,0,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,Female,Bachelor,12887.4317,48767.0,108.0,0,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,Male,Bachelor,7645.8618,0.0,106.0,0,Corporate Auto,SUV,529.881344
4,GA49547,Washington,Male,High School or Below,5363.0765,36357.0,68.0,0,Personal Auto,Four-Door Car,17.269323


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [24]:
customer_data["gender"] = (
    customer_data["gender"]
    .fillna("Unknown")
)

In [25]:
clv_median = customer_data[
    "customer_lifetime_value"
].median()

customer_data["customer_lifetime_value"] = (
    customer_data["customer_lifetime_value"]
    .fillna(clv_median)
)

In [26]:
print("Remaining missing values:")

print(
    customer_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Remaining missing values:
customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64


In [27]:
print("Final Challenge 1 shape:", customer_data.shape)

Final Challenge 1 shape: (9134, 11)


In [28]:
print("Final dataset shape:", customer_data.shape)

print(
    "Completely empty rows:",
    customer_data.isna().all(axis=1).sum()
)

print(
    "Duplicate rows:",
    customer_data.duplicated().sum()
)

print(
    "Total missing values:",
    customer_data.isna().sum().sum()
)

print("\nColumn names:")
print(customer_data.columns.tolist())

Final dataset shape: (9134, 11)
Completely empty rows: 0
Duplicate rows: 0
Total missing values: 0

Column names:
['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']


In [29]:
customer_data.to_csv(
    "customer_data_combined_clean.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


### Challenge 1 Conclusion

The three customer datasets were standardized, combined, and cleaned. We corrected inconsistent column names and category labels, converted customer lifetime value and complaint information into numeric formats, removed empty and duplicate rows, and handled the remaining missing values. The final cleaned dataset was saved for further analysis.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [30]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/data-bootcamp-v4/"
    "data/main/marketing_customer_analysis_clean.csv"
)

marketing_data = pd.read_csv(url)

marketing_data.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [31]:
complaints_long = (
    marketing_data
    .groupby(
        ["policy_type", "month"],
        as_index=False
    )["number_of_open_complaints"]
    .sum()
    .rename(
        columns={
            "number_of_open_complaints": "total_complaints"
        }
    )
)

complaints_long

,policy_type,month,total_complaints
0,Corporate Auto,1,443.434952
1,Corporate Auto,2,385.208135
2,Personal Auto,1,1727.605722
3,Personal Auto,2,1453.684441
4,Special Auto,1,87.074049
5,Special Auto,2,95.226817


In [32]:
complaints_ranked = complaints_long.sort_values(
    by="total_complaints",
    ascending=False
)

complaints_ranked

,policy_type,month,total_complaints
2,Personal Auto,1,1727.605722
3,Personal Auto,2,1453.684441
0,Corporate Auto,1,443.434952
1,Corporate Auto,2,385.208135
5,Special Auto,2,95.226817
4,Special Auto,1,87.074049


In [33]:
highest_by_policy = (
    complaints_long
    .sort_values(
        "total_complaints",
        ascending=False
    )
    .drop_duplicates(subset="policy_type")
    .reset_index(drop=True)
)

highest_by_policy

,policy_type,month,total_complaints
0,Personal Auto,1,1727.605722
1,Corporate Auto,1,443.434952
2,Special Auto,2,95.226817


### Bonus Analysis

The long-format table summarizes the total number of open complaints for every policy type and month combination. Sorting the results by total complaints allows the customer service department to identify the months and policy categories that experienced the highest complaint volumes and may therefore require additional attention.

In [34]:
revenue_by_channel = pd.pivot_table(
    marketing_data,
    index="sales_channel",
    values="customer_lifetime_value",
    aggfunc="sum"
).round(2)

In [35]:
revenue_by_channel = revenue_by_channel.rename(
    columns={
        "customer_lifetime_value": "total_revenue"
    }
)

In [36]:
revenue_by_channel = revenue_by_channel.sort_values(
    by="total_revenue",
    ascending=False
)

revenue_by_channel

,total_revenue
sales_channel,
Agent,33057887.85
Branch,24359201.21
Call Center,17364288.37
Web,12697632.90


### Sales Channel Revenue Analysis

The pivot table compares the total customer lifetime value generated through each sales channel. The channel appearing at the top of the sorted table produced the highest total revenue, while the channel at the bottom produced the lowest.

In [37]:
average_clv = pd.pivot_table(
    marketing_data,
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

average_clv

education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


In [38]:
average_clv["Overall Average"] = (
    average_clv.mean(axis=1).round(2)
)

average_clv

education,Bachelor,College,Doctor,High School or Below,Master,Overall Average
gender,,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05,7956.77
M,7703.60,8052.46,7415.33,8149.69,8168.83,7897.98


### Customer Lifetime Value Analysis

The pivot table compares the average customer lifetime value across gender and education groups. It helps identify which combinations of gender and education have the highest and lowest average customer lifetime value, allowing the marketing department to understand differences between customer segments.

## Final Conclusion

In this lab, we combined and cleaned three customer datasets by standardizing column names and category values, correcting numeric formats, removing empty and duplicate rows, and handling missing values. We then used pivot tables and grouped summaries to compare revenue across sales channels, analyze average customer lifetime value by gender and education, and examine complaint volumes by policy type and month.